# pop scaling sweep -- one-notebook Colab run (resumable)

Runs **both scaling curves** via `scripts/run_scaling.py`, with every artifact stored on
**your Google Drive** so nothing is lost when a Colab session ends:

- **Data-scaling curve** -- arms A (pretrained) & B (no pretraining) x `train_n` in
  {1000, 5000, 15000} x seeds {0, 1}: 12 finetune->generate->eval runs. The 52K (full-data)
  point reuses the earlier `finetune_A_ep10` / `finetune_B_seed{0,1}` runs.
- **Pretrain-compute curve** -- arm A x pretrain-epochs {1, 3}: 2 more runs. The ep10 point
  reuses `finetune_A_ep10` (pretrain-final == epoch-10).

**Prerequisite -- Step 0 pretrain is the first run cell in THIS notebook:** the scaling sweep finetunes
*from* pretrain checkpoints (arm-A data configs load `outputs/pretrain/final`; the
pretrain-compute configs load the stable `outputs/pretrain/epoch-{1,3}` dirs written by
`pop.train.pretrain`'s milestone callback). Those `epoch-{1,3}` dirs come from the milestone callback,
so pretrain must be run with the current code -- the **Step 0 cell below does this
automatically on Run all**, before the preflight/sweep cells. Do **not** substitute a pretrain run from an older
checkout. See `docs/gpu-runbook.md` Step 0 for background.

**Before first run:** create the shared Drive folder `MyDrive/pop_cycle3/` and upload
`pop_repo.zip` into it. The RAG, scaling and LoRA notebooks share this one
workspace, so the zip is uploaded here just once and every arm's results co-locate under a
single `results/` for aggregation. Build the zip locally from your clone of
https://github.com/yib7/pretrain-or-prompt with
`git archive --format=zip -o dist/pop_repo.zip HEAD` (the repo travels to Colab as a
Drive zip rather than a git clone, so no token is needed).

**Every session:** Runtime > Change runtime type > pick a GPU, then Runtime > **Run all**.
Safe to re-run any time -- completed configs are skipped (each finetune/generate/eval step has
a durable done-marker), and progress is always visible in
`Drive/pop_cycle3/logs/scaling/sweep/STATUS.md`.

In [ ]:
import sys

print("Python", sys.version)
assert sys.version_info >= (3, 11), "pop needs Python >= 3.11; this Colab runtime is older"
!nvidia-smi

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/pop_cycle3")
for sub in ("outputs", "results", "logs"):
    (BASE / sub).mkdir(parents=True, exist_ok=True)

ZIP = BASE / "pop_repo.zip"
assert ZIP.is_file(), (
    f"Upload pop_repo.zip to {ZIP.parent} first -- build it locally with\n"
    "  git archive --format=zip -o dist/pop_repo.zip HEAD\n"
    "then drag dist/pop_repo.zip into Drive/pop_cycle3/ (see docs/gpu-runbook.md)."
)
print("Drive workspace ready:", BASE)

In [ ]:
%cd /content
!rm -rf /content/repo
!unzip -q /content/drive/MyDrive/pop_cycle3/pop_repo.zip -d /content/repo
%cd /content/repo
%pip install -q -e .

In [ ]:
# Point the repo's outputs/, results/, logs/ at Drive so checkpoints, predictions, metrics, and
# progress logs all survive session resets. The repo ships a committed results/ directory; its
# contents are copied onto Drive once before the swap. The pretraining checkpoints
# (outputs/pretrain/final and outputs/pretrain/epoch-{1,3}) are produced by Step 0, which you
# run in THIS notebook (the Step 0 cell below) before the sweep -- they land
# under Drive/pop_cycle3/outputs/ so the finetune steps can load them. Do not
# substitute a pretrain run from an older checkout.
import os
import shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/pop_cycle3")
REPO = Path("/content/repo")
for sub in ("outputs", "results", "logs"):
    drive_dir = BASE / sub
    repo_dir = REPO / sub
    if repo_dir.is_symlink():
        repo_dir.unlink()
    elif repo_dir.exists():
        shutil.copytree(repo_dir, drive_dir, dirs_exist_ok=True)
        shutil.rmtree(repo_dir)
    os.symlink(drive_dir, repo_dir)
print("outputs/, results/, logs/ now live on Drive:", BASE)

### Optional: Weights & Biases

The scaling sweep does real T5 finetuning, so W&B *can* log it -- but it is strictly optional
and never on the **Run all** path. `pop` auto-disables W&B unless the `WANDB_API_KEY` env var is
set (see `pop.train.finetune`), so Run-all never stalls waiting for a login. If you want W&B,
add a new cell with `import wandb; wandb.login()` and paste **your own** key when prompted -- the
key is never stored in this notebook or the repo.

## Step 0 -- fresh pretrain (REQUIRED; the sweep finetunes from these checkpoints)

Runs **before** the preflight/sweep cells, in this same session, and **after** the setup
cells 1-4 (which `pip install -e .` the `pop` package). It trains the shared
SentencePiece tokenizer and runs the full span-corruption pretrain, writing
`outputs/tokenizer/tokenizer.model` and the stable
`outputs/pretrain/{final,epoch-1,epoch-3,epoch-10}` dirs that the sweep's arm-A and
pretrain-compute configs load. Skipping it is exactly what raises
`RuntimeError: NOT_FOUND: "outputs/tokenizer/tokenizer.model"` at the first finetune --
the `--list` preflight below only checks config *wiring*, not whether these files exist on
disk.

~1 h on an A100; every artifact lands on `Drive/pop_cycle3/outputs/`. Safe to **Run all**
again: the tokenizer step no-ops if the model already exists, and `pop pretrain`
auto-resumes from the latest `outputs/pretrain/checkpoint-*` on Drive after a disconnect.

In [ ]:
# Step 0 (REQUIRED, runs before the sweep): build the shared SentencePiece tokenizer, then
# pretrain the T5. Writes outputs/tokenizer/tokenizer.model and the stable
# outputs/pretrain/{final,epoch-1,epoch-3,epoch-10} dirs the sweep's finetune steps load.
#
# NOTE: both steps call `pop` via ! (subprocess), NOT `import pop`. The `pip install -e .` in
# the setup cell above is an *editable* install -- visible to fresh subprocesses but not to
# this already-running kernel until a restart, so an in-kernel `import pop` would raise
# `No module named 'pop'`. Run cells 1-4 first (or Runtime > Run all) so `pop` is installed.
#
# Tokenizer training is CPU (a few min, incl. corpus download); `pop pretrain` uses the GPU
# (~1 h on an A100) and auto-resumes from the latest outputs/pretrain/checkpoint-* on Drive.
import os

if os.path.exists("outputs/tokenizer/tokenizer.model"):
    print("tokenizer already present -- skipping tokenizer training")
else:
    !pop tokenizer --out outputs/tokenizer/tokenizer.model --vocab-size 16384 --corpus-samples 50000 --seed 42

!pop pretrain --config configs/pretrain_10ep.yaml

# Confirm the artifacts the sweep needs exist before continuing to the preflight/sweep.
!ls -d outputs/tokenizer/tokenizer.model outputs/pretrain/final outputs/pretrain/epoch-1 outputs/pretrain/epoch-3

In [ ]:
# Quick CPU-only sanity check: print the 42-step plan (14 configs) and run preflight without
# touching a GPU.
!python scripts/run_scaling.py --list

In [ ]:
# The long-running cell. Re-running after any interruption continues where it left off: each
# config's finetune (best/model.safetensors), generate (best/predictions_test.jsonl), and eval
# (results/*.json) step is skipped once its artifact exists.
!python scripts/run_scaling.py

In [ ]:
import json
from pathlib import Path

status = Path("logs/scaling/sweep/STATUS.md")
if status.exists():
    print(status.read_text(encoding="utf-8"))
summary = Path("logs/scaling/sweep/SUMMARY.md")
if summary.exists():
    print(summary.read_text(encoding="utf-8"))
for path in sorted(Path("results").glob("finetune_scale_*_test.json")) + sorted(
    Path("results").glob("finetune_ptcompute_*_test.json")
):
    metrics = json.loads(path.read_text(encoding="utf-8"))["metrics"]
    print(
        f"{path.name}: CodeBLEU={metrics['codebleu']:.4f} "
        f"syntax={metrics['syntax_valid_rate']:.4f} EM={metrics['em']:.4f} n={metrics['n']}"
    )

### If the session disconnects or hits the GPU quota

Normal and expected on the free tier. Reopen this notebook and **Run all** again: finished
finetune/generate/eval steps are skipped instantly and the sweep resumes at the first config
without a `best/model.safetensors` / `predictions_test.jsonl` / results file; a partially-
trained finetune resumes from its latest `checkpoint-*`. Progress at any time:
`Drive/pop_cycle3/logs/scaling/sweep/STATUS.md`.